In [6]:
import torch
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments,T5Tokenizer,T5ForConditionalGeneration
from datasets import load_dataset

In [2]:
dataset = load_dataset('supremezxc/nlpcc_2017')
dataset

DatasetDict({
    train: Dataset({
        features: ['version', 'data'],
        num_rows: 50000
    })
})

In [3]:
dataset = dataset['train'].train_test_split(100,seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['version', 'data'],
        num_rows: 49900
    })
    test: Dataset({
        features: ['version', 'data'],
        num_rows: 100
    })
})

In [4]:
dataset['train'][0]

{'version': '0.0.1',
 'data': {'content': '发布日期:2014-12-2708:25:36【字体:】预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。图例标准防御指南12小时内可能出现能见度小于500米的雾,或者已经出现能见度小于500米、大于等于200米的雾并将持续。1、有关部门和单位按照职责做好防雾准备工作;2、机场、高速公路、轮渡码头等单位加强交通管理,保障安全;3、驾驶人员注意雾的变化,小心驾驶;4、户外活动注意安全。',
  'title': '松原市发布大雾黄色预警:预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。...'}}

In [10]:
dataset['train'][0]['data']['content']

'发布日期:2014-12-2708:25:36【字体:】预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。图例标准防御指南12小时内可能出现能见度小于500米的雾,或者已经出现能见度小于500米、大于等于200米的雾并将持续。1、有关部门和单位按照职责做好防雾准备工作;2、机场、高速公路、轮渡码头等单位加强交通管理,保障安全;3、驾驶人员注意雾的变化,小心驾驶;4、户外活动注意安全。'

In [5]:
tokenizer = T5Tokenizer.from_pretrained('Langboat/mengzi-t5-base')
tokenizer

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5Tokenizer(name_or_path='Langboat/mengzi-t5-base', vocab_size=32028, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<e

In [16]:
dataset['train']['data']

[{'content': '发布日期:2014-12-2708:25:36【字体:】预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。图例标准防御指南12小时内可能出现能见度小于500米的雾,或者已经出现能见度小于500米、大于等于200米的雾并将持续。1、有关部门和单位按照职责做好防雾准备工作;2、机场、高速公路、轮渡码头等单位加强交通管理,保障安全;3、驾驶人员注意雾的变化,小心驾驶;4、户外活动注意安全。',
  'title': '松原市发布大雾黄色预警:预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。...'},
 {'content': '菲律宾副总统比奈7月1日在马卡迪举行大型政治集会,猛烈“炮轰”阿基诺政府。【环球网报道记者聂鲁彬】“懒惰、迟缓且优柔寡断。”菲律宾总统阿基诺日前遭到来自其副手在数千人面前近乎吊打式的责骂。据《菲律宾日报》7月2日报道,菲律宾副总统杰约马尔•比奈1日在其家族影响力最强的马卡迪市发起政治集会,在讲话中他向阿基诺发出最为尖锐攻击。在这场用菲律宾语发表的言辞激烈、感情洋溢的讲话中,比奈在15分钟里多次质问“政府跑到哪里去了”。“阿基诺执政五年过去了,大批国民仍然没有工作、忍受着饥饿与疾病,且无处求助。很多年轻人没法上学,我们看到的是盗匪横行、毒品泛滥、贫困蔓延。”比奈严厉鞭笞政府的失职:“整个国家在问:政府跑到哪里去了?”此外,比奈还批评政府对1月25日发生的马马萨帕诺村武装冲突无所作为。菲律宾44名特警在那场与“摩洛伊斯兰解放阵线”的冲突中丧生。另有17名武装分子和5名平民死亡。比奈称,至今没有一名凶手遭到起诉。“笨拙、麻木”这样的词汇,再次出现在1日的集会上。比奈继最早于上周用这些词语解释他与阿基诺政府分道扬镳的决定。他对参与集会的支持者承诺,其所属的菲律宾行动联盟(UNA)“不会与那些笨拙、麻木”的人为伍,并表示UNA假若犯了什么错误会爽快承认,“我们不会推诿或责怪他人,我们会带着羞愧感承担责任”。在上周与阿基诺政府公开裂痕后,比奈的此番讲话被普遍认为是为其2016年参与大选奠定基调。比奈强调,如果他获选总统,将推进社会福利、农业、制造业和旅游业的发展,同时改善高质量基础教育和医疗保障的水平。',
  'title': '菲副总统比奈发起政治集会,称阿基诺执政后盗匪横行、毒品泛滥、贫困蔓延,炮轰政府

In [22]:
def process_func(examples):
    examples = examples['data']
    contents = ['摘要生成：\n'+ e['content'] for e in examples]
    titles = [e['title'] for e in examples]
    inputs = tokenizer(contents,max_length=384,truncation=True)
    labels = tokenizer(text_target=titles,max_length=64,truncation=True)
    inputs['labels'] = labels['input_ids']
    
    return inputs


In [23]:
tokenized_dataset = dataset.map(process_func,batched=True)
tokenized_dataset

Map:   0%|          | 0/49900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['version', 'data', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 49900
    })
    test: Dataset({
        features: ['version', 'data', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [27]:
print(tokenizer.decode(tokenized_dataset['train'][0]['input_ids']))

摘要生成: 发布日期:2014-12-2708:25:36【字体:】预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。图例标准防御指南12小时内可能出现能见度小于500米的雾,或者已经出现能见度小于500米、大于等于200米的雾并将持续。1、有关部门和单位按照职责做好防雾准备工作;2、机场、高速公路、轮渡码头等单位加强交通管理,保障安全;3、驾驶人员注意雾的变化,小心驾驶;4、户外活动注意安全。</s>


In [28]:
print(tokenizer.decode(tokenized_dataset['train'][0]['labels']))

松原市发布大雾黄色预警:预计未来12小时,我市部分地方有雾,能见度较差,请注意预防。...</s>


In [29]:
model = T5ForConditionalGeneration.from_pretrained('Langboat/mengzi-t5-base')

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [47]:
#评估指标
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()


def compute_metric(evalpred):
    predictions,labels = evalpred
    decode_preds = tokenizer.batch_decode(predictions,skip_special_tokens=True)
    labels = np.where(labels !=-100,labels,tokenizer.pad_token_id)
    '''
    DataCollatorForSeq2Seq 在 训练时 会自动将 labels 中的 [PAD] token 替换为 -100，用于 mask 掉这些 padding 位，不参与 loss 计算。
    '''
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decode_preds = [p.strip() for p in decode_preds]
    decode_labels = [l.strip() for l in decode_labels]
    
    
    
    scores = rouge.get_scores(decode_preds,decode_labels,avg=True)
    
    return {
        "rouge-1": scores["rouge-1"]["f"],
        "rouge-2": scores["rouge-2"]["f"],
        "rouge-l": scores["rouge-l"]["f"],
    }

In [48]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    logging_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="rouge-l",
    predict_with_generate=True
)

In [49]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_dataset["train"].select(range(50)),
    eval_dataset=tokenized_dataset["test"].select(range(50)),
    compute_metrics=compute_metric,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer)
)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_38849/4075605886.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l
0,No log,7.086804,0.000000,0.000000,0.000000
1,No log,5.902100,0.000000,0.000000,0.000000
2,No log,5.461797,0.000000,0.000000,0.000000


TrainOutput(global_step=3, training_loss=8.734030405680338, metrics={'train_runtime': 656.5264, 'train_samples_per_second': 0.228, 'train_steps_per_second': 0.005, 'total_flos': 57000699924480.0, 'train_loss': 8.734030405680338, 'epoch': 2.6153846153846154})

In [52]:
from transformers import pipeline
pipe = pipeline('text2text-generation',model = model,tokenizer=tokenizer,device = 0)

Device set to use mps:0


In [53]:
pipe('摘要生成：\n'+dataset["test"][-1]['data']["content"],max_length=64,do_sample=True)

[{'generated_text': ',警德,“在在。杀到杀行为犯罪侵害刑,死:在死犯非法犯罪责任追究不法,,杀。 走,走有对死杀刑刑犯走走非法人走下“死,出,发生,死亡团伙破坏过。被杀,'}]

In [55]:
dataset["test"][-1]['data']["content"]

'南方都市报2014-11-2800:00南都讯在市场里听见有人喊“抓贼”,麦某明等人闻声追上去,抓住一名男子拳打脚踢,致其倒地颅脑损伤,最终不治身亡。中山市第一人民法院一审认定麦某明犯故意伤害罪,判处有期徒刑12年,麦某明不服,他认为自己和被害人素不相识,属于见义勇为,目前已经向中山市中级人民法院提出上诉。[案情经过]被打5个月后死亡悲剧发生在2013年9月19日早上6时许。中山市第一人民法院查明,当天早上,吴某胜将装了菜的三轮车停在西区沙朗市场附近,后来他看见一名年约60岁的男子触碰三轮车上的菜,遂大喊“抓贼”。随后,吴某胜以及他旁边的麦某明、潘某松一起追赶这名男子,直至西区沙朗爱意商场附近,该男子被抓住。法院查明,麦某明、吴某胜接着对该男子拳打脚踢,造成他受伤,送医院后抢救无效,于今年2月7日死亡。经法医鉴定,被害人何某康,因头部外伤致特重型颅脑损伤,继发肺部感染、慢性消耗性恶液质、多器官功能衰竭死亡。2013年9月29日、10月1日,警方分别将麦某明、吴某胜抓获。公诉人指控被告人麦某明、吴某胜涉嫌故意伤害罪。庭审过程中,被告人麦某明推翻之前的供述,否认对被害人何某康实施了殴打。两打人者获刑罚法庭上,公诉人提交了案发现场的监控录像,证明案发时有三名男子追逐一名男子(被害人何某康),其中一名男子追上后没有明显殴打动作,后面两名男子追上后连续对何某康殴打。“我听到抓贼才去追赶的,我的行为是见义勇为。我与被害人无冤无仇,也并不相识”,法庭上,被告人麦某明辩称,他只是抓住何某康的手不让他去打人,他还被何某康打了腹部,何某康后来自己倒地了。中山市第一人民法院审理认为,被告人麦某明、吴某胜结伙故意伤害他人身体,致一人死亡,其行为已构成故意伤害罪。被告人吴某胜归案后如实供述其罪行,依法可以从轻处罚。一审法院判处被告人麦某明犯故意伤害罪,判处有期徒刑12年,剥夺政治权利4年。判处被告人吴有胜犯故意伤害罪,判处有期徒刑11年,剥夺政治权利3年。麦某明不服一审判决,已经向中山市中级人民法院提出上诉。案件普法点在本案中,麦某明自认为是出于见义勇为,却酿成了惨痛的后果。那么麦某明的行为是否属于见义勇为?在量刑中能不能有所体现?听闻“抓小偷”应该围观还是冲上前?与不法行为作斗争应该鼓励,但不应殴打根据《广东省见义勇为人员奖励和保障条例》规定,见义勇为是指不负有法定职责、法定义务的人员,

In [54]:
dataset['test'][-1]['data']['title']

'中山:路人帮忙追打疑似小偷,致其死亡被判12年;不服上诉,自认属见义勇为。'